In [2]:
import torch
import pickle
import logging
import os , re
import numpy as np
import librosa
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset , DataLoader
from audio_visual import visual , audio

In [ ]:
# class Net(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.hidden_dim = 128
#         self.action_dim = 4
#         self.output_dim = 4
#         self.x_dim = 36
#         self.y_dim = 128
#         self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#         self.visualnet = visual.VisualNet(64 , 4 ,width_dim=128 , height_dim=128).to(self.device)
#         v_model_path = "./audio_visual/visual_weights_1.pth"
#         self.visualnet.load_state_dict(torch.load(v_model_path))
#         self.visualnet.eval()
#         a_model_path = "./checkpoint/audio_weights_fine_tune.pth"
#         self.audio_net = audio.AudioNet(64 , 4 ,width_dim=128 , height_dim=36).to(self.device)
#         self.audio_net.load_state_dict(torch.load(a_model_path))
#         self.audio_net.eval()
#         self.q_net = nn.Sequential(
#             nn.Linear(self.hidden_dim + 1, self.hidden_dim),
#             nn.ReLU(),
#             nn.Linear(self.hidden_dim, self.hidden_dim),
#             nn.ReLU(),
#             nn.Linear(self.hidden_dim, 1)
#         )
#         self.value_net = nn.Sequential(
#             nn.Linear(self.hidden_dim, self.hidden_dim),
#             nn.ReLU(),
#             nn.Linear(self.hidden_dim, self.hidden_dim),
#             nn.ReLU(),
#             nn.Linear(self.hidden_dim, 1)
#         )
#         self.policy_net = nn.Sequential(
#             nn.Linear(self.hidden_dim, self.hidden_dim),
#             nn.ReLU(),
#             nn.Linear(self.hidden_dim, self.hidden_dim),
#             nn.ReLU(),
#             nn.Linear(self.hidden_dim, self.action_dim)
#         )
#         self.initialize_weights_uniform()
#     def forward(self, audio, visual_input,labes):
#         labes = labes.unsqueeze(1)
#         batch ,height , width , _ = visual_input.shape
#         result = self.audio_net(audio)
#         x = torch.max(result,dim=1).indices
#         mask = list()
#         for i in x:
#             mask_ = np.ones((height, width),dtype=np.uint8)
#             height , width = mask_.shape
#             if i.item() == 1:
#                 for h_i in range(height):
#                     for w_i in range(width):
#                     # left
#                         if h_i + 2* w_i < 128:
#                             mask_[h_i][w_i] = 0
#             if i.item() == 2:
#                 for h_i in range(height):
#                     for w_i in range(width):
#                         # right
#                         if h_i - 2* w_i < -128:
#                             mask_[h_i][w_i] = 0
#             if  i.item() == 0:
#                    for h_i in range(height):
#                     for w_i in range(width):
#                         # mid
#                         if h_i - 2* w_i > -128 and h_i + 2* w_i > 128:
#                             mask_[h_i][w_i] = 0
#             mask_ = mask_[:, :, np.newaxis]
#             mask.append(mask_)
#         mask = torch.from_numpy(np.array(mask)).to(self.device)
#         red_overlay = np.array([255.0, 0.0, 0.0 , 1], dtype=np.float32)
#         overlay = torch.from_numpy(red_overlay).to(self.device)
#         # visual = overlay*(1-mask)*visual + mask*visual
#         visual = visual_input*mask
#         visual = visual.permute(0,3,2,1)
#         combined_encode = self.visualnet.visual_net(visual)
#         q_input_dim = torch.cat((combined_encode , labes) , dim=1)
#         Q = self.q_net(q_input_dim)
#         V = self.value_net(combined_encode)
#         P = self.policy_net(combined_encode)
#         return Q,V,P
#     def initialize_weights_uniform(self, weight_range=(-0.1, 0.1), bias_range=(-0.1, 0.1)):
#         for name, module in self.named_modules():
#             if isinstance(module, (nn.Linear, nn.Conv2d)):
#                 # 初始化权重为均匀分布
#                 if hasattr(module, 'weight') and module.weight is not None:
#                     nn.init.uniform_(module.weight, a=weight_range[0], b=weight_range[1])
#                 # 初始化偏置为均匀分布
#                 if hasattr(module, 'bias') and module.bias is not None:
#                     nn.init.uniform_(module.bias, a=bias_range[0], b=bias_range[1])

In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_dim = 128
        self.action_dim = 4
        self.output_dim = 4
        self.x_dim = 36
        self.y_dim = 128
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.visualnet = visual.VisualNet(64 , 4 ,width_dim=128 , height_dim=128).to(self.device)
        v_model_path = "./audio_visual/visual_weights_1.pth"
        self.visualnet.load_state_dict(torch.load(v_model_path))
        self.visualnet.eval()
        a_model_path = "./checkpoint/audio_weights_fine_tune.pth"
        self.audio_net = audio.AudioNet(64 , 4 ,width_dim=128 , height_dim=36).to(self.device)
        self.audio_net.load_state_dict(torch.load(a_model_path))
        self.audio_net.eval()
        self.v_net = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, 16)
        )
        self.policy_net = nn.Sequential(
            nn.Linear(20, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, 4)
        )
        self.q_net = nn.Sequential(
            nn.Linear(self.hidden_dim+1, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, 1)
        )
        self.initialize_weights_uniform()
    def forward(self, audio, visual_input,labes):
        labes = labes.unsqueeze(1)
        batch ,height , width , _ = visual_input.shape
        result = self.audio_net(audio)
        x = torch.max(result,dim=1).indices
        mask = list()
        for i in x:
            mask_ = np.ones((height, width),dtype=np.uint8)
            height , width = mask_.shape
            if i.item() == 1:
                for h_i in range(height):
                    for w_i in range(width):
                    # left
                        if h_i + 2* w_i < 128:
                            mask_[h_i][w_i] = 0
            if i.item() == 2:
                for h_i in range(height):
                    for w_i in range(width):
                        # right
                        if h_i - 2* w_i < -128:
                            mask_[h_i][w_i] = 0
            if  i.item() == 0:
                   for h_i in range(height):
                    for w_i in range(width):
                        # mid
                        if h_i - 2* w_i > -128 and h_i + 2* w_i > 128:
                            mask_[h_i][w_i] = 0
            mask_ = mask_[:, :, np.newaxis]
            mask.append(mask_)
        mask = torch.from_numpy(np.array(mask)).to(self.device)
        red_overlay = np.array([255.0, 0.0, 0.0 , 1], dtype=np.float32)
        overlay = torch.from_numpy(red_overlay).to(self.device)
        # visual = overlay*(1-mask)*visual + mask*visual
        visual = visual_input*mask
        visual = visual.permute(0,3,2,1)
        combined_encode = self.visualnet.visual_net(visual)
        q_input_dim = torch.cat((combined_encode , labes) , dim=1)
        Q = self.q_net(q_input_dim)
        # V = self.value_net(combined_encode)
        # P = self.policy_net(combined_encode)
        return Q
    def initialize_weights_uniform(self, weight_range=(-0.1, 0.1), bias_range=(-0.1, 0.1)):
        for name, module in self.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                # 初始化权重为均匀分布
                if hasattr(module, 'weight') and module.weight is not None:
                    nn.init.uniform_(module.weight, a=weight_range[0], b=weight_range[1])
                # 初始化偏置为均匀分布
                if hasattr(module, 'bias') and module.bias is not None:
                    nn.init.uniform_(module.bias, a=bias_range[0], b=bias_range[1])

In [5]:
class MyData(Dataset):
    def __init__(self,data):
        self.audio_ = self.get_data(data,'audio')
        self.tag_ = self.get_data(data,'rl_pred')
        self.visual_ = self.get_data(data,'camera')
        self.reward_ = self.get_data(data,'reward')
        self.audio = list()
        self.tag = list()
        self.visual = list()
        self.reward = list()
        for index,tag in enumerate(self.tag_):
            if tag == 3:
                self.audio_ = self.audio_[:index]
                self.visual_ = self.visual_[:index]
                self.reward_ = self.reward_[:index]
                self.tag_ = self.tag_[:index]
                break
        logging.info(self.tag_)
    def get_data(self , data , name):
        d = list()
        for i in range(len(data)):
            d.extend(data[i][name])
        return d
    def __len__(self):
        return len(self.audio_)
    def __getitem__(self,index):
        return self.audio_[index] ,self.visual_[index] ,self.tag_[index] ,self.reward_[index]

In [6]:
class IQL:
    def __init__(self, model, learning_rate=1e-5):
        self.gamma = 0.99
        self.model = model
        self.device = model.device
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
    
    def compute_loss(self, Q, target_Q, labels):
        target_Q = target_Q.unsqueeze(1)
        q_loss = F.mse_loss(Q, target_Q)
        return q_loss

    def train_step(self, audio, visual_input, labels, target_Q):
        Q= self.model(audio, visual_input, labels)
        q_loss= self.compute_loss(Q ,target_Q, labels)
        self.optimizer.zero_grad()
        q_loss.backward()
        self.optimizer.step()

        return q_loss.item()

In [12]:
from torch.utils.tensorboard import SummaryWriter
log_dir = './logs/rl2'
writer = SummaryWriter(log_dir)

In [14]:
path = "../data/RL/random"
files = os.listdir(path=path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net().to(device)
iql = IQL(model)
num_epochs = len(files)
for epoch in range(num_epochs):
    model.train()
    e = 0
    q = 0
    file_path = os.path.join(path, files[epoch])
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    dataset = MyData(data)
    dataloader = DataLoader(dataset=dataset , batch_size=16 ,shuffle=True)
    for batch_idx, batch in enumerate(dataloader):
        e+=batch_idx
        batch_audio, batch_visual, batch_labels  ,batch_reward = batch
        batch_audio, batch_visual, batch_labels  ,batch_reward= batch_audio.to(device), batch_visual.to(device), batch_labels.to(device) ,batch_reward.to(device)

        q_loss= iql.train_step(batch_audio,batch_visual, batch_labels, batch_reward)
        q+=q_loss
    if e ==0 :
        pass
    q_loss = q/e
    writer.add_scalar('Loss/train', q_loss, epoch)
    print(f'Episode {epoch},q_loss {q_loss}')

Episode 0,q_loss 249.59911390940348
Episode 1,q_loss 96.12930946350097
Episode 2,q_loss 26.899558448791502
Episode 3,q_loss 79.15840193430583
Episode 4,q_loss 446.1552673339844
Episode 5,q_loss 73.39281728050925
Episode 6,q_loss 51.43292114951394
Episode 7,q_loss 108.34454366895888
Episode 8,q_loss 302.60354904901413
Episode 9,q_loss 26.647347259521485
Episode 10,q_loss 61.56609356042111
Episode 11,q_loss 34.761945608890414
Episode 12,q_loss 100.73971184624565
Episode 13,q_loss 25.042106691996256
Episode 14,q_loss 36.54879389053736
Episode 15,q_loss 65.71642858331853
Episode 16,q_loss 390.12266540527344
Episode 17,q_loss 77.14510761607777
Episode 18,q_loss 25.09563700358073
Episode 19,q_loss 41.12330951112689
Episode 20,q_loss 38.61366771351207
Episode 21,q_loss 31.71288466020064
Episode 22,q_loss 19.7389222462972
Episode 23,q_loss 209.80507562810723
Episode 24,q_loss 55.18261552290483
Episode 25,q_loss 43.226719538370766
Episode 26,q_loss 17.564516003926595
Episode 27,q_loss 44.829788

ZeroDivisionError: float division by zero